<a href="https://colab.research.google.com/github/dalexandre963-cmd/Optimizing-Metabolite-Production-by-Manipulating-TF-Expression/blob/main/MCF24_TF_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimizing Metabolite Production by Manipulating TF Expression

In this Python Notebook, we use regulatory information from YEASTRACT+ to simulate TF KOs for optimizing a given cell product.

We use the Yeast 8.5.0 metabolic model, considering a minimal and a rich medium.

- [Pedro T. Monteiro](https://github.com/ptgm) - INESC-ID/IST
- [Diogo A. Couceiro](https://github.com/dalexandre963-cmd) - IBB/IST

---
## Installation

In [ ]:
# To install COBRApy on Google Colab, uncomment the following line
!pip3 install cobra
# To install COBRApy on local machine run this on the terminal
# pip3 install cobra
import cobra

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 27.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.8/117.8 kB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.7/526.7 kB 40.3 MB/s eta 0:00:00


If you have another model, replace yeast8GEM.xml with the appropriate model filename

In [ ]:
# Fetch the model file (Yeast 8.5.0 GEM - S.cerevisiae) from server
!wget -c 'http://pedromonteiro.org/models/yeast8GEM.xml'

--2024-05-12 16:41:25--  http://pedromonteiro.org/models/yeast8GEM.xml
Resolving pedromonteiro.org (pedromonteiro.org)... 213.36.252.182, 213.36.252.183, 2a01:e0d:1:2:58bf:f9b7:0:1, ...
Connecting to pedromonteiro.org (pedromonteiro.org)|213.36.252.182|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://arsr.inesc-id.pt/~ptgm/models/yeast8GEM.xml [following]
--2024-05-12 16:41:26--  http://arsr.inesc-id.pt/~ptgm/models/yeast8GEM.xml
Resolving arsr.inesc-id.pt (arsr.inesc-id.pt)... 146.193.36.1
Connecting to arsr.inesc-id.pt (arsr.inesc-id.pt)|146.193.36.1|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12794002 (12M) [text/xml]
Saving to: ‘yeast8GEM.xml’

yeast8GEM.xml       100%[===================>]  12.20M  6.18MB/s    in 2.0s    

2024-05-12 16:41:29 (6.18 MB/s) - ‘yeast8GEM.xml’ saved [12794002/12794002]



---
## Loading the metabolic model

In [ ]:
# Loading Yeast8GEM model twice: One for each medium
modelVerduynMedium = cobra.io.read_sbml_model('yeast8GEM.xml')
modelRichMedium = cobra.io.read_sbml_model('yeast8GEM.xml')

# Set a timeout for some KOs that never end...
modelVerduynMedium.solver.configuration.timeout = 20
modelRichMedium.solver.configuration.timeout = 20

modelRichMedium

Name,M_yeastGEM_v8__46__4__46__2
Memory address,7aecf735ab90
Number of metabolites,2742
Number of reactions,4058
Number of genes,1150
Number of groups,90
Objective expression,1.0*r_2111 - 1.0*r_2111_reverse_58b69
Compartments,"cell envelope, cytoplasm, extracellular, mitochondrion, nucleus, peroxisome, endoplasmic reticulum, Golgi, lipid particle, vacuole, endoplasmic reticulum membrane, vacuolar membrane, Golgi membrane, mitochondrial membrane"


---
## Defining the Medium
If you don't define the medium, all the analysis are defined using the minimal medium already defined in the metabolic model.

### Reaction/bounds identification
We start by trying to identify the reaction ID for each of the exchange reactions that we are interested in specifying the bounds.

In [ ]:
for r in modelRichMedium.exchanges:
    if 'glucose' in r.name.lower():
        print(r.id, r.name, r.reaction, r.bounds)

r_1714 D-glucose exchange s_0565[e] <=>  (-1.0, 1000.0)
r_4502 D-glucose 6-phosphate exchange s_4150[e] -->  (0.0, 1000.0)
r_4504 D-glucose 1-phosphate exchange s_4145[e] -->  (0.0, 1000.0)


### Setting the medium
Once we note all the desired reactions, we can actually change the medium, for each of those reactions. Here we create two mediums: minimal and rich. One for each model loaded previously.

In [ ]:
import tabulate
# Verduyn minimal medium
medium = modelVerduynMedium.medium # Values in mmol/gDW/h
medium['r_1654'] = 1000 # ammonium exchange
medium['r_1714'] = 1.15 # glucose exchange
medium['r_1992'] = 1000 # oxygen exchange
medium['r_1832'] = 1000 # H+ exchange
medium['r_1861'] = 1000 # iron(2+) exchange
medium['r_2005'] = 1000 # phosphate exchange
medium['r_2020'] = 1000 # potassium exchange
medium['r_2049'] = 1000 # sodium exchange
medium['r_2060'] = 1000 # sulfate exchange
medium['r_4593'] = 1000 # chloride exchange
medium['r_4594'] = 1000 # Cu2(+) exchange
medium['r_4595'] = 1000 # Mn(2+) exchange
medium['r_4596'] = 1000 # Zn(2+) exchange
medium['r_4597'] = 1000 # Mg(2+) exchange
medium['r_4600'] = 1000 # Ca(2+) exchange
modelVerduynMedium.medium = medium

# Rich medium
medium = modelRichMedium.medium # Values in mmol/gDW/h
medium['r_1654'] = 1000 # ammonium exchange
medium['r_1714'] = 80   # glucose exchange
medium['r_1992'] = 2    # oxygen exchange
medium['r_1832'] = 1000 # H+ exchange
medium['r_1861'] = 1000 # iron(2+) exchange
medium['r_2005'] = 1000 # phosphate exchange
medium['r_2020'] = 1000 # potassium exchange
medium['r_2049'] = 1000 # sodium exchange
medium['r_2060'] = 1000 # sulfate exchange
medium['r_4593'] = 1000 # chloride exchange
medium['r_4594'] = 1000 # Cu2(+) exchange
medium['r_4595'] = 1000 # Mn(2+) exchange
medium['r_4596'] = 1000 # Zn(2+) exchange
medium['r_4597'] = 1000 # Mg(2+) exchange
medium['r_4600'] = 1000 # Ca(2+) exchange
modelRichMedium.medium = medium

tab = [['ReactionID', 'ReactionName', 'Verduyn', 'Rich']]
for rid in modelVerduynMedium.medium.keys():
    r = modelVerduynMedium.reactions.get_by_id(rid)
    tab.append([r.id, r.name, modelVerduynMedium.medium[rid], modelRichMedium.medium[rid]])
tabulate.tabulate(tab, headers='firstrow', tablefmt='html')

ReactionID,ReactionName,Verduyn,Rich
r_1654,ammonium exchange,1000,1000
r_1714,D-glucose exchange,1.15,80
r_1832,H+ exchange,1000,1000
r_1861,iron(2+) exchange,1000,1000
r_1992,oxygen exchange,1000,2
r_2005,phosphate exchange,1000,1000
r_2020,potassium exchange,1000,1000
r_2049,sodium exchange,1000,1000
r_2060,sulphate exchange,1000,1000
r_2100,water exchange,1000,1000


---
## WT solution

In [ ]:
solWTVerduyn = cobra.flux_analysis.pfba(modelVerduynMedium)
print('Reference values for the WT solution (Verduyn medium):')
print('  status:', solWTVerduyn.status)
print('  biomass:', solWTVerduyn.objective_value)
print('  ethanol:', solWTVerduyn.fluxes['r_1761'])

Reference values for the WT solution (Verduyn medium):
  status: optimal
  biomass: 116.15416485978677
  ethanol: 0.0


In [ ]:
solWTRich = cobra.flux_analysis.pfba(modelRichMedium)
print('Reference values for the WT solution (Rich medium):')
print('  status:', solWTRich.status)
print('  biomass:', solWTRich.objective_value)
print('  ethanol:', solWTRich.fluxes['r_1761'])

Reference values for the WT solution (Rich medium):
  status: optimal
  biomass: 2574.9701817592586
  ethanol: 130.95199555548027


---
## Load YEASTRACT data

Here, we load the regulations file from YEASTRACT, concerning a specific Yeast

In [ ]:
yeastractfile = 'Yeastract-Sc-regs.tsv' # CHANGE the species file here !

# Fetch the regulations file (S.cerevisiae) from server
!wget -c "http://pedromonteiro.org/models/$yeastractfile"

--2024-05-12 16:41:50--  http://pedromonteiro.org/models/Yeastract-Sc-regs.tsv
Resolving pedromonteiro.org (pedromonteiro.org)... 213.36.252.182, 213.36.252.183, 2a01:e0d:1:2:58bf:f9b7:0:1, ...
Connecting to pedromonteiro.org (pedromonteiro.org)|213.36.252.182|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://arsr.inesc-id.pt/~ptgm/models/Yeastract-Sc-regs.tsv [following]
--2024-05-12 16:41:50--  http://arsr.inesc-id.pt/~ptgm/models/Yeastract-Sc-regs.tsv
Resolving arsr.inesc-id.pt (arsr.inesc-id.pt)... 146.193.36.1
Connecting to arsr.inesc-id.pt (arsr.inesc-id.pt)|146.193.36.1|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 65368676 (62M) [text/tab-separated-values]
Saving to: ‘Yeastract-Sc-regs.tsv’

Yeastract-Sc-regs.t 100%[===================>]  62.34M  11.2MB/s    in 6.5s    

2024-05-12 16:41:57 (9.60 MB/s) - ‘Yeastract-Sc-regs.tsv’ saved [65368676/65368676]



### Process the regulations file

In [ ]:
dTmp = {}
fh = open(yeastractfile, 'r')
for line in fh:
    fields = line.strip().split('\t')
    if fields[1] not in dTmp:
       dTmp[fields[1]] = { 'orf': fields[0], 'dTGs': {},
            'sPos': set(), 'sNeg': set(), 'sIndir': set(), 'sDir': set()}
    db_orf, gene, association, code = fields[2], fields[3], fields[5], fields[6]

    # If used model is iRV781, then the ORF needs to be parsed
    # since the ORF in YEASTRACT is different from the ORF in the model
    # orf = 'CAALFM_' + db_orf[:2] + db_orf[3:-2] + db_orf[-1:]
    # else use this line
    orf = db_orf
    if orf not in dTmp[fields[1]]['dTGs']:
        dTmp[fields[1]]['dTGs'][orf] = {'gene': gene, 'db_orf': db_orf}
    if code=='Indirect': dTmp[fields[1]]['sIndir'].add(orf)
    elif code=='Direct': dTmp[fields[1]]['sDir'].add(orf)
    if association=='Positive':   dTmp[fields[1]]['sPos'].add(orf)
    elif association=='Negative': dTmp[fields[1]]['sNeg'].add(orf)

dTFs = {}
for orfTF in dTmp:
    if orfTF not in dTFs:
        dTFs[orfTF] = {'orf': dTmp[orfTF]['orf'], 'dTGs': {},
            'sBEp': set(), 'sBEn': set(), 'sEp': set(), 'sEn': set()}
    for orf in dTmp[orfTF]['dTGs']:
        if orf in dTmp[orfTF]['sDir'] and orf not in dTmp[orfTF]['sIndir']:
            continue # ignore the Direct only
        if orf in dTmp[orfTF]['sPos'] and orf in dTmp[orfTF]['sNeg']:
            continue # ignore the Dual
        if orf not in dTmp[orfTF]['sPos'] and orf not in dTmp[orfTF]['sNeg']:
            continue # ignore the N/A
        if orf in dTmp[orfTF]['sIndir']:
            dTFs[orfTF]['dTGs'][orf] = dTmp[orfTF]['dTGs'][orf]
            if orf in dTmp[orfTF]['sDir']:
                if orf in dTmp[orfTF]['sPos']: dTFs[orfTF]['sBEp'].add(orf)
                if orf in dTmp[orfTF]['sNeg']: dTFs[orfTF]['sBEn'].add(orf)
            else:
                if orf in dTmp[orfTF]['sPos']: dTFs[orfTF]['sEp'].add(orf)
                if orf in dTmp[orfTF]['sNeg']: dTFs[orfTF]['sEn'].add(orf)


---
## Optimization of a given compound with Under/Over-Expression

Reference: https://repositorium.sdum.uminho.pt/handle/1822/23370

### Auxiliar methods: to parse Boolean expressions

In [ ]:
!pip3 install lark
from lark import Lark, Transformer, v_args

gene_rule_grammar = """
    ?start: fors
          | genename "=" value -> assign_var
    ?fors: lors                -> f_l_ors
    ?lors: fands               -> b_s
        | lors "or" fands      -> b_l
    ?fands: lands              -> f_l_ands
    ?lands: atom               -> b_s
        | lands "and" atom     -> b_l
    ?value: NUMBER             -> number
    ?atom: genename            -> var
         | "(" fors ")"
         | "reset"             -> reset_vars
    genename: (DIGIT|LETTER|"-"|"_")+
    %import common.DIGIT
    %import common.LETTER
    %import common.NUMBER
    %import common.WS_INLINE
    %ignore WS_INLINE
"""

@v_args(inline=True)    # Affects the signatures of the methods
class EvaluateTree(Transformer):
    number = float

    def __init__(self):
        self.reset_vars()

    def b_l(self, args, arg):
        return args + [ arg ]
    def b_s(self, arg):
        return [ arg ]

    def f_l_ands(self, largs):
        return min(largs)
    def f_l_ors(self, largs):
        return sum(largs) / len(largs)

    def var(self, name):
        return self.vars[name] if name in self.vars else 1
    def assign_var(self, name, value):
        self.vars[name] = value
        return value
    def reset_vars(self):
        self.vars = {}
        return 0

gene_rule_parser = Lark(gene_rule_grammar, parser='lalr', transformer=EvaluateTree())
rule_eval = gene_rule_parser.parse

def parseRule(rules2p, sRule):
  sRule = sRule.strip()
  if sRule in rules2p: # previously computed
    return rules2p[sRule]
  if not sRule: # no rule -> no change in reaction
    p = 1
  else:
    p = rule_eval(sRule)
  rules2p[sRule] = p
  return rules2p[sRule]


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 2.4 MB/s eta 0:00:00


### Run the actual experiment

Considering a Parametrization with the:
- TF name
- Experiments (pairs Under/Over-expression)
- Model: Verduyn or Rich

In [ ]:
userTF = 'PDC2' # <- REPLACE HERE your TF of interest

tExps = ((0.00,1.25), (0.00,1.50),\
         (0.25,1.25), (0.25,1.50),\
         (0.50,1.25), (0.50,1.50),\
         (1.25,0.00), (1.25,0.25), (1.25,0.50),\
         (1.50,0.00), (1.50,0.25), (1.50,0.50))

# Select the correct model HERE
# model = modelVerduynMedium
model = modelRichMedium

# Select the correct reverence WT solution HERE
# solWT = solWTVerduyn
solWT = solWTRich

In [ ]:
dExpTGs = {
    'BEp':(dTFs[userTF]['sBEp'],[]),
    'BEn':([],dTFs[userTF]['sBEn']),
    'BEpn':(dTFs[userTF]['sBEp'],dTFs[userTF]['sBEn']),
    'Ep':(dTFs[userTF]['sEp'],[]),
    'En':([],dTFs[userTF]['sEn']),
    'Epn':(dTFs[userTF]['sEp'],dTFs[userTF]['sEn'])}

# Store all flux information in a dictionary
dFlux = { 'WT': { r.id: solWT.fluxes[r.id] for r in model.reactions } }

# Iterate over all planned experiments
for k,sORFs in dExpTGs.items():
  print('-------------------------------------------------------------------')
  print('Simulating TF:', userTF, '  Targets', k, ' #TGs: ', len(sORFs[0]) + len(sORFs[1]))
  print(' ',len(sORFs[0]),'Positives:', ' '.join(sORFs[0]))
  print(' ',len(sORFs[1]),'Negatives:', ' '.join(sORFs[1]))

  if len(sORFs[0])==0 and len(sORFs[1])==0:
    continue # optimization: no TG to perturb

  for exp in tExps:
    # print('   Exp:', exp, ' exp[0]:',exp[0], ' exp[1]:', exp[1], flush=True)
    dFlux[exp] = {} # create first or clean previous

    # Setup the Experiment
    dGeneModifs = {}
    for orf in sORFs[0]:
      dGeneModifs[dTFs[userTF]['dTGs'][orf]['db_orf']] = {'value': exp[0], 'orf': orf}
    if exp[0]>=0: # Not a KO
      for orf in sORFs[1]:
        dGeneModifs[dTFs[userTF]['dTGs'][orf]['db_orf']] = {'value': exp[1], 'orf': orf}

    # Run Experiment and generate results
    with model as myModelExp:
      if exp[0]<0: # convention: -1 is a KO
        print('KO should not happen in TF mode !!!!!')
        for orf_db in dGeneModifs: # multiple KO
          myModelExp.genes.get_by_id(dGeneModifs[orf]['orf']).knock_out()
      else:
        #---- begin Under/Over Expression
        rules2p = {}
        rule_eval('reset')
        for orf_db in dGeneModifs: # set values \forall dGeneModifs
          rule_eval(dGeneModifs[orf_db]['orf'] + '=' + str(dGeneModifs[orf_db]['value']))
        for r in myModelExp.reactions:
          if not any([dGeneModifs[orf_db]['orf'] in r.gene_reaction_rule for orf_db in dGeneModifs]):
            continue # optimization: reaction \not contains any targeted gene
          wtflux = solWT.fluxes[r.id]
          if wtflux==0: # keep bounds (see Ref paper)
            continue
          p = parseRule(rules2p, r.gene_reaction_rule)
          newbound = p * wtflux
          if p == 1: # optimization: no change was performed
            continue
          elif p > 1: # Overexpression
            if wtflux > 0: # Pos ref: p.vi < flux < UB
              r.lower_bound = min(newbound, r.upper_bound)
            else: # Neg ref: LB < flux < p.vi
              r.upper_bound = max(newbound, r.lower_bound)
          else: # Underexpression
            if wtflux > 0: # Pos ref: 0 < flux < p.vi
              r.lower_bound = 0
              r.upper_bound = newbound
            else: # Neg ref: p.vi < flux < 0
              r.lower_bound = newbound
              r.upper_bound = 0
      #---- end Under/Over expression
      try:
        sol = cobra.flux_analysis.pfba(myModelExp) # pFBA
        if sol==None or sol.status=='infeasible' or \
           sol.fluxes['r_1761'] <= solWT.fluxes['r_1761']:
          print('  Exp:', exp, '  NotBetter   biomass:', sol.objective_value, '  ethanol:', sol.fluxes['r_1761'], flush=True)
          del dFlux[exp]
        else:
          print('  Exp:', exp, '  status:', sol.status, '  biomass:', sol.objective_value, '  ethanol:', sol.fluxes['r_1761'], flush=True)
          dFlux[exp] = { r.id: sol.fluxes[r.id] for r in model.reactions }
      except:
        print('  Exp:', exp, '  infeasible', flush=True)
        del dFlux[exp]

  if len(dFlux.keys())==1 and 'WT' in dFlux.keys():
      continue # optimization: no experiment was performed
  print('Saving results to file: Fluxes_' + userTF + '_' + k + '.csv')
  # Save dFlux results from exp to file
  with open('Fluxes_' + userTF + '_' + k + '.csv', 'w') as fh:
    fh.write('Reaction ID;Reaction Name;Gene Rule;Reaction IDs;Reaction Names' + ''.join([';'+str(exp)+';'+str(exp) for exp in dFlux]) + '\n')
    for r in model.reactions:
      # write the reaction string to file
      fh.write(r.id + ';' + r.name + ';' + r.gene_reaction_rule + ';' + r.reaction + ';' + r.build_reaction_string(use_metabolite_names=True))
      for e in dFlux:
        if r.id in dFlux[e]:
          fh.write(';' + str(dFlux[e][r.id]) + ';' + str(dFlux[e][r.id]-dFlux['WT'][r.id]))
        else:
          fh.write(';;')
      fh.write('\n')
  print()


-------------------------------------------------------------------
Simulating TF: PDC2   Targets BEp  #TGs:  4
  4 Positives: YGR144W YOL055C YLR134W YBR092C
  0 Negatives: 
  Exp: (0.0, 1.25)   NotBetter   biomass: 1645.4337470240705   ethanol: 87.1970884117034
  Exp: (0.0, 1.5)   NotBetter   biomass: 1645.4337470240705   ethanol: 87.1970884117034
  Exp: (0.25, 1.25)   NotBetter   biomass: 1876.2324854427927   ethanol: 98.09672446316628
  Exp: (0.25, 1.5)   NotBetter   biomass: 1876.2324854427927   ethanol: 98.09672446316628
  Exp: (0.5, 1.25)   NotBetter   biomass: 2107.6041838253177   ethanol: 108.99636051462919
  Exp: (0.5, 1.5)   NotBetter   biomass: 2107.6041838253177   ethanol: 108.99636051462919
  Exp: (1.25, 0.0)   status: optimal   biomass: 2576.1259962127133   ethanol: 138.09035017208373
  Exp: (1.25, 0.25)   status: optimal   biomass: 2576.1259962127133   ethanol: 138.09035017208373
  Exp: (1.25, 0.5)   status: optimal   biomass: 2576.1259962127133   ethanol: 138.090350172

If a given TF in the context of some Experiment gave a better result, a result.csv file was generated. Download it, and open it as a SpreadSheet.